# Naive Bayes Spell Correction — LinguaQuest
Training model dari data typo vocab QuestData, lalu export parameter ke JSON.

**Alur:**
1. Prepare training data
2. Train Naive Bayes model
3. Simpan model ke `.pkl`
4. Export parameter model ke `.json` (untuk Flutter)

In [9]:
import pickle
import json
import math
from collections import defaultdict

## 1. Training Data
Pasangan (typo → correct) berdasarkan 50 vocab aktual QuestData Level 1–10.

In [10]:
training_data = [
    # Level 1: Basic Greetings
    ('helo',       'Hello'),
    ('helllo',     'Hello'),
    ('hllo',       'Hello'),
    ('hlelo',      'Hello'),
    ('goodby',     'Goodbye'),
    ('gooodbye',   'Goodbye'),
    ('godbye',     'Goodbye'),
    ('godbey',     'Goodbye'),
    ('plaese',     'Please'),
    ('plese',      'Please'),
    ('plase',      'Please'),
    ('pelase',     'Please'),
    ('thank yu',   'Thank you'),
    ('thnk you',   'Thank you'),
    ('thnak you',  'Thank you'),
    ('thank yuo',  'Thank you'),
    ('sorr',       'Sorry'),
    ('sory',       'Sorry'),
    ('soryr',      'Sorry'),
    ('soory',      'Sorry'),

    # Level 2: Food & Drinks
    ('aple',       'Apple'),
    ('aplle',      'Apple'),
    ('appel',      'Apple'),
    ('appl',       'Apple'),
    ('watr',       'Water'),
    ('watter',     'Water'),
    ('watre',      'Water'),
    ('wtaer',      'Water'),
    ('braed',      'Bread'),
    ('brea',       'Bread'),
    ('breead',     'Bread'),
    ('bredd',      'Bread'),
    ('coffe',      'Coffee'),
    ('cofee',      'Coffee'),
    ('coffie',     'Coffee'),
    ('cofffe',     'Coffee'),
    ('rce',        'Rice'),
    ('riice',      'Rice'),
    ('risce',      'Rice'),
    ('ric',        'Rice'),

    # Level 3: Numbers & Colors
    ('erd',        'Red'),
    ('redd',       'Red'),
    ('re',         'Red'),
    ('rdee',       'Red'),
    ('bule',       'Blue'),
    ('bleu',       'Blue'),
    ('bloo',       'Blue'),
    ('bluw',       'Blue'),
    ('grene',      'Green'),
    ('gren',       'Green'),
    ('geren',      'Green'),
    ('greeen',     'Green'),
    ('yello',      'Yellow'),
    ('yelow',      'Yellow'),
    ('yellw',      'Yellow'),
    ('yelolw',     'Yellow'),
    ('whiet',      'White'),
    ('whit',       'White'),
    ('wihte',      'White'),
    ('whte',       'White'),

    # Level 4: Family
    ('moter',      'Mother'),
    ('mothr',      'Mother'),
    ('mther',      'Mother'),
    ('mothre',     'Mother'),
    ('fathe',      'Father'),
    ('fathr',      'Father'),
    ('fther',      'Father'),
    ('fahter',     'Father'),
    ('brothe',     'Brother'),
    ('brothr',     'Brother'),
    ('brthr',      'Brother'),
    ('broher',     'Brother'),
    ('sistr',      'Sister'),
    ('siter',      'Sister'),
    ('sisstre',    'Sister'),
    ('siste',      'Sister'),
    ('frend',      'Friend'),
    ('freind',     'Friend'),
    ('firend',     'Friend'),
    ('friand',     'Friend'),

    # Level 5: Travel
    ('airprt',     'Airport'),
    ('airpot',     'Airport'),
    ('aiport',     'Airport'),
    ('arport',     'Airport'),
    ('htel',       'Hotel'),
    ('hotl',       'Hotel'),
    ('hotle',      'Hotel'),
    ('hoel',       'Hotel'),
    ('tickt',      'Ticket'),
    ('tiket',      'Ticket'),
    ('tcket',      'Ticket'),
    ('tciket',     'Ticket'),
    ('mpa',        'Map'),
    ('maap',       'Map'),
    ('mapp',       'Map'),
    ('ampp',       'Map'),
    ('passprt',    'Passport'),
    ('passort',    'Passport'),
    ('pasprt',     'Passport'),
    ('passoprt',   'Passport'),

    # Level 6: Work & Office
    ('meetng',     'Meeting'),
    ('meting',     'Meeting'),
    ('meeeting',   'Meeting'),
    ('meeing',     'Meeting'),
    ('deadlin',    'Deadline'),
    ('dedline',    'Deadline'),
    ('deadlne',    'Deadline'),
    ('deedline',   'Deadline'),
    ('reprt',      'Report'),
    ('repot',      'Report'),
    ('repotr',     'Report'),
    ('repport',    'Report'),
    ('mangr',      'Manager'),
    ('mangaer',    'Manager'),
    ('managr',     'Manager'),
    ('maneger',    'Manager'),
    ('salry',      'Salary'),
    ('salayr',     'Salary'),
    ('slary',      'Salary'),
    ('sallary',    'Salary'),

    # Level 7: Health
    ('doctr',      'Doctor'),
    ('docter',     'Doctor'),
    ('docotr',     'Doctor'),
    ('dcotor',     'Doctor'),
    ('medicne',    'Medicine'),
    ('medcine',    'Medicine'),
    ('mediicne',   'Medicine'),
    ('medicin',    'Medicine'),
    ('hosptal',    'Hospital'),
    ('hosptial',   'Hospital'),
    ('hospiatl',   'Hospital'),
    ('hospitl',    'Hospital'),
    ('fevr',       'Fever'),
    ('fveer',      'Fever'),
    ('fevre',      'Fever'),
    ('feer',       'Fever'),
    ('excerise',   'Exercise'),
    ('exrecise',   'Exercise'),
    ('exercse',    'Exercise'),
    ('exerise',    'Exercise'),

    # Level 8: Technology
    ('computr',    'Computer'),
    ('copmuter',   'Computer'),
    ('compuet',    'Computer'),
    ('compter',    'Computer'),
    ('internt',    'Internet'),
    ('intrenet',   'Internet'),
    ('internnet',  'Internet'),
    ('interent',   'Internet'),
    ('passwrd',    'Password'),
    ('passord',    'Password'),
    ('pasword',    'Password'),
    ('paswrod',    'Password'),
    ('downlad',    'Download'),
    ('doenload',   'Download'),
    ('downlaod',   'Download'),
    ('downlod',    'Download'),
    ('sofware',    'Software'),
    ('softwre',    'Software'),
    ('sftware',    'Software'),
    ('softwear',   'Software'),

    # Level 9: Nature & Environment
    ('mountan',    'Mountain'),
    ('moutain',    'Mountain'),
    ('moutnain',   'Mountain'),
    ('mountian',   'Mountain'),
    ('ocaen',      'Ocean'),
    ('ocen',       'Ocean'),
    ('oocean',     'Ocean'),
    ('oceon',      'Ocean'),
    ('forset',     'Forest'),
    ('frest',      'Forest'),
    ('forrest',    'Forest'),
    ('forets',     'Forest'),
    ('rivr',       'River'),
    ('rvier',      'River'),
    ('rivre',      'River'),
    ('rievr',      'River'),
    ('weathr',     'Weather'),
    ('wetaher',    'Weather'),
    ('weathe',     'Weather'),
    ('wheater',    'Weather'),

    # Level 10: Advanced Conversation
    ('negotate',    'Negotiate'),
    ('negociate',   'Negotiate'),
    ('negotiat',    'Negotiate'),
    ('negotaite',   'Negotiate'),
    ('colaborate',  'Collaborate'),
    ('collabrate',  'Collaborate'),
    ('colaborte',   'Collaborate'),
    ('collaboarte', 'Collaborate'),
    ('inovative',   'Innovative'),
    ('innovatve',   'Innovative'),
    ('inoovative',  'Innovative'),
    ('innovatif',   'Innovative'),
    ('perspectve',  'Perspective'),
    ('persepctive', 'Perspective'),
    ('perspektiv',  'Perspective'),
    ('perspectiv',  'Perspective'),
    ('acomplish',   'Accomplish'),
    ('accomplsh',   'Accomplish'),
    ('acomplsh',    'Accomplish'),
    ('accompish',   'Accomplish'),
]

print(f'Total training samples: {len(training_data)}')

Total training samples: 200


## 2. Feature Extraction
Dua fitur yang dipakai model:
- **Length difference**: selisih panjang typo vs correct word
- **Char overlap**: jumlah karakter unik yang sama (set intersection)

In [11]:
def count_char_overlap(a: str, b: str) -> int:
    """Hitung karakter unik yang sama antara dua string (case-insensitive)."""
    return len(set(a.lower()) & set(b.lower()))

def extract_features(typo: str, correct: str) -> dict:
    """Ekstrak fitur dari satu pasangan (typo, correct)."""
    return {
        'length_diff':   abs(len(typo) - len(correct)),
        'char_overlap':  count_char_overlap(typo, correct),
    }

# Preview fitur
sample = training_data[0]
print(f'Typo: "{sample[0]}" → Correct: "{sample[1]}", Features: {extract_features(*sample)}')

Typo: "helo" → Correct: "Hello", Features: {'length_diff': 1, 'char_overlap': 4}


## 3. Training Naive Bayes Model
Hitung:
- `prior[word]` = P(word)
- `length_likelihood[word][diff]` = P(length_diff | word)
- `char_likelihood[word][overlap]` = P(char_overlap | word)

Semua menggunakan **Laplace smoothing** agar tidak ada probabilitas nol.

In [12]:
class NaiveBayesSpellModel:
    def __init__(self):
        self.prior             = {}   # P(correct_word)
        self.length_likelihood = {}   # P(length_diff | correct_word)
        self.char_likelihood   = {}   # P(char_overlap | correct_word)
        self.word_counts       = {}   # raw count per word
        self.total_samples     = 0

    def train(self, data: list[tuple]):
        word_count    = defaultdict(int)
        length_counts = defaultdict(lambda: defaultdict(int))
        char_counts   = defaultdict(lambda: defaultdict(int))

        for typo, correct in data:
            key = correct.lower()
            word_count[key] += 1

            feats = extract_features(typo, correct)
            length_counts[key][feats['length_diff']] += 1
            char_counts[key][feats['char_overlap']]  += 1

        total = len(data)
        self.total_samples = total
        self.word_counts   = dict(word_count)

        # Prior probability P(word)
        for word, count in word_count.items():
            self.prior[word] = count / total

        # Likelihood dengan Laplace smoothing
        for word, count in word_count.items():
            # P(length_diff | word) — smoothing +1/+5
            self.length_likelihood[word] = {
                diff: (c + 1) / (count + 5)
                for diff, c in length_counts[word].items()
            }
            # P(char_overlap | word) — smoothing +1/+10
            self.char_likelihood[word] = {
                overlap: (c + 1) / (count + 10)
                for overlap, c in char_counts[word].items()
            }

        print(f'✅ Training selesai! {len(word_count)} kata, {total} samples.')

    def predict(self, user_input: str, correct_word: str, threshold: float = -10.0) -> str | None:
        """
        Prediksi apakah user_input adalah typo dari correct_word.
        Return correct_word jika typo terdeteksi, None jika bukan typo.
        """
        input_lower   = user_input.lower()
        correct_lower = correct_word.lower()

        if input_lower == correct_lower:
            return None  # sudah benar

        feats = extract_features(input_lower, correct_lower)

        # Log prior
        prior = self.prior.get(correct_lower, 1 / (len(self.prior) + 1))
        log_prob = math.log(prior)

        # Log likelihood: length_diff
        length_prob = self.length_likelihood.get(correct_lower, {}).get(feats['length_diff'], 0.1)
        log_prob += math.log(length_prob)

        # Log likelihood: char_overlap
        char_prob = self.char_likelihood.get(correct_lower, {}).get(feats['char_overlap'], 0.05)
        log_prob += math.log(char_prob)

        return correct_word if log_prob > threshold else None


# Train model
model = NaiveBayesSpellModel()
model.train(training_data)

✅ Training selesai! 50 kata, 200 samples.


## 4. Evaluasi Model
Uji model pada training data dan beberapa contoh typo baru (unseen).

In [13]:
print('=== Evaluasi pada Training Data (sample 10) ===')
for typo, correct in training_data[:10]:
    result = model.predict(typo, correct)
    status = '✅' if result == correct else '❌'
    print(f'{status} Input: "{typo}" → Prediksi: {result} (expected: {correct})')

print('\n=== Evaluasi pada Typo Baru (unseen) ===')
unseen_tests = [
    ('hellp',     'Hello'),
    ('gooodby',   'Goodbye'),
    ('aplpe',     'Apple'),
    ('wateer',    'Water'),
    ('freiend',   'Friend'),
    ('compuuter', 'Computer'),
    ('xyz123',    'Computer'),    # bukan typo → harus None
    ('abcdef',    'Hello'),       # bukan typo → harus None
]
for typo, correct in unseen_tests:
    result = model.predict(typo, correct)
    print(f'Input: "{typo}" → {result}')

=== Evaluasi pada Training Data (sample 10) ===
✅ Input: "helo" → Prediksi: Hello (expected: Hello)
✅ Input: "helllo" → Prediksi: Hello (expected: Hello)
✅ Input: "hllo" → Prediksi: Hello (expected: Hello)
✅ Input: "hlelo" → Prediksi: Hello (expected: Hello)
✅ Input: "goodby" → Prediksi: Goodbye (expected: Goodbye)
✅ Input: "gooodbye" → Prediksi: Goodbye (expected: Goodbye)
✅ Input: "godbye" → Prediksi: Goodbye (expected: Goodbye)
✅ Input: "godbey" → Prediksi: Goodbye (expected: Goodbye)
✅ Input: "plaese" → Prediksi: Please (expected: Please)
✅ Input: "plese" → Prediksi: Please (expected: Please)

=== Evaluasi pada Typo Baru (unseen) ===
Input: "hellp" → Hello
Input: "gooodby" → Goodbye
Input: "aplpe" → Apple
Input: "wateer" → Water
Input: "freiend" → Friend
Input: "compuuter" → Computer
Input: "xyz123" → Computer
Input: "abcdef" → Hello


In [17]:
# Tambahkan di cell evaluasi untuk debug score
print('=== Debug Score ===')
test_cases = [
    ('helo',    'Hello'),      # typo ringan → harusnya terdeteksi
    ('xyz',     'Hello'),      # salah jauh → harusnya None
    ('abcdef',  'Hello'),      # salah jauh → harusnya None
    ('appllle', 'Apple'),      # typo ringan → harusnya terdeteksi
    ('qwerty',  'Apple'),      # salah jauh → harusnya None
]
for typo, correct in test_cases:
    feats = extract_features(typo.lower(), correct.lower())
    prior = model.prior.get(correct.lower(), 0.01)
    lp = math.log(prior)
    lp += math.log(model.length_likelihood.get(correct.lower(), {}).get(feats['length_diff'], 0.1))
    lp += math.log(model.char_likelihood.get(correct.lower(), {}).get(feats['char_overlap'], 0.05))
    print(f'"{typo}" → "{correct}": score={lp:.4f}')

=== Debug Score ===
"helo" → "Hello": score=-5.9757
"xyz" → "Hello": score=-9.2103
"abcdef" → "Hello": score=-7.7187
"appllle" → "Apple": score=-7.4674
"qwerty" → "Apple": score=-8.0064


## 5. Simpan Model ke `.pkl`

In [15]:
with open('spell_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print('✅ Model disimpan ke spell_model.pkl')

✅ Model disimpan ke spell_model.pkl


## 6. Export Parameter Model ke JSON (untuk Flutter)
Flutter tidak bisa baca `.pkl` langsung, jadi kita export parameter model
(prior + likelihood) ke format `.json` yang bisa di-load sebagai Flutter asset.

In [16]:
# Load dari pkl untuk memastikan konsistensi
with open('spell_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Konversi key int → string (JSON hanya support string key)
def int_keys_to_str(d: dict) -> dict:
    return {str(k): v for k, v in d.items()}

model_json = {
    'metadata': {
        'model_type':     'naive_bayes_spell_correction',
        'version':        '1.0.0',
        'total_samples':  loaded_model.total_samples,
        'vocab_size':     len(loaded_model.prior),
        'threshold':      -6.5,
    },
    'prior': loaded_model.prior,
    'length_likelihood': {
        word: int_keys_to_str(dist)
        for word, dist in loaded_model.length_likelihood.items()
    },
    'char_likelihood': {
        word: int_keys_to_str(dist)
        for word, dist in loaded_model.char_likelihood.items()
    },
}

with open('spell_model.json', 'w') as f:
    json.dump(model_json, f, indent=2)

print('✅ Model diekspor ke spell_model.json')
print(f'   Vocab size  : {model_json["metadata"]["vocab_size"]} kata')
print(f'   Total sample: {model_json["metadata"]["total_samples"]}')
print(f'   Keys sample : {list(model_json["prior"].keys())[:5]}')

✅ Model diekspor ke spell_model.json
   Vocab size  : 50 kata
   Total sample: 200
   Keys sample : ['hello', 'goodbye', 'please', 'thank you', 'sorry']
